# Preference Fine Tuning- DPO demo on a T4 GPU: annotated walkthrough

**Goal:** teach a small language model (Qwen2.5-0.5B-Instruct) to prefer better answers using **DPO (Direct Preference Optimization)**, then compare it with the original model.

**DPO in one line:** show the model pairs of answers (a *chosen* one and a *rejected* one) and nudge it to make the chosen kind more likely, while a frozen copy of the original model (the "reference") keeps it from drifting too far.

**Roadmap**
1. Check the GPU
2. Install tooling, look at the default DPO config
3. Prepare the preference dataset
4. First successful training run (no checkpoint saved)
5. Re-run with checkpoint saving
6. Compare base vs DPO answers
7. Read the results

**About the ⚠️ cells:** they are real problems hit on Colab/T4, kept because they explain why the final command looks the way it does. You can skip them.

**To reproduce from scratch:** set the runtime to **T4 + High-RAM**, then run Steps 1, 2, 3, 5, 6, 7 in order (Step 4 is the same training without saving the model).

## Step 1: Check the hardware
Confirms what we have: a Tesla T4 (15 GB memory, compute capability 7.5), Python 3.13 and CUDA 13.

**T4 limits that shape everything below:** no native bf16 support and only 15 GB of GPU memory, so we train in float32 and pick a small 0.5B-parameter model.

In [1]:
# STEP 1: show GPU model, memory, driver, compute capability, Python, CUDA and PyTorch (bf16 flag)
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv;
python --version; nvcc --version | tail -2;
python -c "import torch;print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_capability(),
'bf16:', torch.cuda.is_bf16_supported())"

name, memory.total [MiB], driver_version, compute_cap
Tesla T4, 15360 MiB, 580.82.07, 7.5
Python 3.13.16
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
2.11.0+cu130 True (7, 5) bf16: True


## Step 2: Install uv, view the default DPO config
NeMo launches everything through **uv** (a fast Python package manager) that builds isolated environments. We clone the repo and print the default DPO config (examples/configs/dpo.yaml).

Settings worth knowing (the output is long; scroll to the end for data, logger and cluster):
- `dpo.reference_policy_kl_penalty` (0.05): the "leash" keeping the model near the reference. This is beta in the DPO paper.
- `dpo.max_num_steps`: how many training steps to run.
- `policy.model_name`: which model to fine-tune.
- `policy.train_global_batch_size`: preference pairs per step.
- `data.*`: where the preference data comes from.

In [6]:
# STEP 2: install uv, clone NeMo RL (shallow), print the sub-repo list and the default DPO config
!pip -q install uv && git clone -q --depth 1 https://github.com/NVIDIA-NeMo/RL.git nemo-rl && cd nemo-rl && git log -1 --format=%h && cat .gitmodules && echo ===== && cat examples/configs/dpo.yaml

6705e9d
[submodule "3rdparty/Megatron-Bridge"]
	path = 3rdparty/Megatron-Bridge-workspace/Megatron-Bridge
	url = https://github.com/NVIDIA-NeMo/Megatron-Bridge.git
	branch = main
	shallow = true
[submodule "3rdparty/Automodel-workspace/Automodel"]
	path = 3rdparty/Automodel-workspace/Automodel
	url = https://github.com/NVIDIA-NeMo/Automodel.git
	branch = r0.6.0
	shallow = true
[submodule "3rdparty/Gym-workspace/Gym"]
	path = 3rdparty/Gym-workspace/Gym
	url = https://github.com/NVIDIA-NeMo/Gym.git
	branch = main
	shallow = true
=====
# DPO Algorithm Configuration
dpo:
  max_num_epochs: 1
  max_num_steps: 150
  # Optional checkpoint boundary for time-sliced jobs. The final scheduler
  # horizon remains max_num_steps across resumed segments.
  stop_after_step: null
  val_period: 25
  val_batches: 8
  val_global_batch_size: 8
  val_micro_batch_size: 1
  val_at_start: true
  val_at_end: false
  seed: 42

  reference_policy_kl_penalty: 0.05
  preference_average_log_probs: False # whether nor

## Step 3: Pull submodules and build the preference dataset
NeMo depends on a few bundled sub-repos (Automodel, Megatron-Bridge, Gym), so we fetch them. Then we build a tiny dataset from **Intel/orca_dpo_pairs**: 512 training rows and 88 validation rows, one JSON object per line:

`{"prompt": ..., "chosen": ..., "rejected": ...}`

In this dataset the "chosen" answers come from GPT-4 (detailed, step by step) and the "rejected" ones from a weaker model. That is why DPO pushes the model's *style* toward longer, more careful answers. NeMo RL reads this format with its BinaryPreferenceDataset class.

In [7]:
# STEP 3: fetch sub-repos, then write 512 train + 88 validation preference pairs to /content/*.jsonl
!cd /content/nemo-rl && git submodule update --init --depth 1 --recursive 2>&1 | tail -4; cd /content &&
python -c "import json,datasets as d; ds=d.load_dataset('Intel/orca_dpo_pairs',split='train').shuffle(seed=0).select(range(600)); [open(f,'w').write(''.join(json.dumps({'prompt':r['question'],'chosen':r['chosen'],'rejected':r['rejected']})+'\n' for r in ds.select(rng))) for f,rng in (('train.jsonl',range(512)),('val.jsonl',range(512,600)))]" && wc -l train.jsonl val.jsonl && head -c 400 train.jsonl

Cloning into '/content/nemo-rl/3rdparty/Megatron-Bridge-workspace/Megatron-Bridge/3rdparty/Megatron-LM'...
From https://github.com/NVIDIA/Megatron-LM
 * branch            3e95097b4a9ab6ea6b2a39eb2b2b24e4a6dfa54a -> FETCH_HEAD
Submodule path '3rdparty/Megatron-Bridge-workspace/Megatron-Bridge/3rdparty/Megatron-LM': checked out '3e95097b4a9ab6ea6b2a39eb2b2b24e4a6dfa54a'
    512 train.jsonl
     88 val.jsonl
    600 total
{"prompt": "I am trying to decide whether it's worth it to invest in this film proposal. Can you help me answer a few questions? If you can't, please say \"No I can't\".  Question: What is Ted Lewis wife's name? Movie title: Alien Trespass Movie plot: It was a summer night in 1957 in the quiet California desert town of Mojave. Local astronomer Ted Lewis (Eric McCormack) is making a special anniver

### ⚠️ Failed attempt 1: matplotlib backend
Crashed at import. Colab sets the MPLBACKEND environment variable to its own inline notebook backend, which does not exist inside NeMo RL's isolated environment.
**Fix:** prefix the command with `MPLBACKEND=Agg`.

In [3]:
!cd /content/nemo-rl && uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=false data.train.dataset_name=BinaryPreferenceDataset data.train.data_path=/content/train.jsonl data.train.split=train data.validation.dataset_name=BinaryPreferenceDataset data.validation.data_path=/content/val.jsonl data.validation.split=train data.default.dataset_name=BinaryPreferenceDataset data.default.prompt_key=prompt data.default.chosen_key=chosen data.default.rejected_key=rejected 2>&1 | tee /content/dpo.log

/bin/bash: line 1: cd: /content/nemo-rl: No such file or directory


### ⚠️ Failed attempt 2: new config keys need ++
Hydra (the config tool NeMo RL uses) refuses to override a key that is not already in the default YAML, such as `data.train.data_path`.
**Fix:** prefix those keys with `++` (force-add), e.g. `++data.train.data_path=/content/train.jsonl`.

In [5]:
!cd /content/nemo-rl && MPLBACKEND=Agg uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=false data.train.dataset_name=BinaryPreferenceDataset data.train.data_path=/content/train.jsonl data.train.split=train data.validation.dataset_name=BinaryPreferenceDataset data.validation.data_path=/content/val.jsonl data.validation.split=train data.default.dataset_name=BinaryPreferenceDataset data.default.prompt_key=prompt data.default.chosen_key=chosen data.default.rejected_key=rejected 2>&1 | tee /content/dpo.log

/bin/bash: line 1: cd: /content/nemo-rl: No such file or directory


### ⚠️ Failed attempt 3: out of RAM on the standard Colab runtime
The config was now valid and the model started loading, but the standard runtime has only about 12 GB of CPU RAM. At startup NeMo RL's worker compiles CUDA kernels using many parallel compiler processes, and Ray's memory monitor killed the worker.

In [8]:
!cd /content/nemo-rl && MPLBACKEND=Agg uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=false ++data.train.dataset_name=BinaryPreferenceDataset ++data.train.data_path=/content/train.jsonl ++data.train.split=train ++data.validation.dataset_name=BinaryPreferenceDataset ++data.validation.data_path=/content/val.jsonl ++data.validation.split=train ++data.default.dataset_name=BinaryPreferenceDataset ++data.default.prompt_key=prompt ++data.default.chosen_key=chosen ++data.default.rejected_key=rejected 2>&1 | tee /content/dpo.log

Using CPython 3.13.14
Creating virtual environment `nemo-rl-cp3.13.14-6703181c57679eb7`
Installed 266 packages in 1m 28s
INFO:nemo_rl.transformers_compat:transformers_compat.py:70: Installed the Transformers compatibility helper is_torch_fx_available for legacy remote model code
2026-10-24 04:39:47,037	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
INFO:nemo_rl.distributed.virtual_cluster:virtual_cluster.py:547: Started local cluster with tag 'nrl_tag_ALL': {'CPU': 2.0, 'nrl_tag_ALL': 1.0, 'node:172.28.0.12': 1.0, 'memory': 9257738650.0, 'accelerator_type:T4': 1.0, 'GPU': 1.0, 'object_store_memory': 3967602278.0, 'node:__internal_head__': 1.0}
Loaded configuration from: /content/nemo-rl/examples/configs/dpo.yaml
Overrides: ['policy.model_name=Qwen/Qwen2.5-0.5B-Instruct', 'policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct', 'policy.precision=float32', 'policy.train_global_batch_size=8', 'policy.train_micro_batch_size=1', 'policy.max_tota

### ⚠️ Failed attempt 4: adding swap space
Tried to create a 16 GB swap file as extra memory. Colab's container does not allow it (`swapon failed: Invalid argument`).

In [9]:
!free -h; (fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile) 2>&1 | tail -3; free -h; nvidia-smi --query-gpu=memory.used --format=csv; pkill -f ray; sleep 2; df -h /content | tail -1

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.7Gi       7.0Gi       3.8Mi       3.2Gi         9Gi
Swap:             0B          0B          0B
Setting up swapspace version 1, size = 16 GiB (17179865088 bytes)
no label, UUID=d3746f95-1f47-41b3-ae42-45b66240bdaf
swapon: /swapfile: swapon failed: Invalid argument
               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.7Gi       7.0Gi       3.8Mi       3.2Gi         9Gi
Swap:             0B          0B          0B
memory.used [MiB]
0 MiB
^C


### ⚠️ Failed attempt 5: turning off Ray's memory monitor
Disabled Ray's out-of-memory killer with `RAY_memory_monitor_refresh_ms=0`. The worker then died anyway at model load, so the machine really is too small.
**Conclusion:** switch the runtime to **High-RAM** (about 50 GB).

In [10]:
!rm -f /swapfile; cd /content/nemo-rl && RAY_memory_monitor_refresh_ms=0 MPLBACKEND=Agg uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=false ++data.train.dataset_name=BinaryPreferenceDataset ++data.train.data_path=/content/train.jsonl ++data.train.split=train ++data.validation.dataset_name=BinaryPreferenceDataset ++data.validation.data_path=/content/val.jsonl ++data.validation.split=train ++data.default.dataset_name=BinaryPreferenceDataset ++data.default.prompt_key=prompt ++data.default.chosen_key=chosen ++data.default.rejected_key=rejected 2>&1 | tee /content/dpo2.log

INFO:nemo_rl.transformers_compat:transformers_compat.py:70: Installed the Transformers compatibility helper is_torch_fx_available for legacy remote model code
2026-10-24 04:51:59,321	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
INFO:nemo_rl.distributed.virtual_cluster:virtual_cluster.py:547: Started local cluster with tag 'nrl_tag_ALL': {'accelerator_type:T4': 1.0, 'GPU': 1.0, 'node:172.28.0.12': 1.0, 'node:__internal_head__': 1.0, 'nrl_tag_ALL': 1.0, 'object_store_memory': 3987635404.0, 'CPU': 2.0, 'memory': 9304482612.0}
/root/.cache/uv/environments-v2/nemo-rl-cp3.13.14-6703181c57679eb7/lib/python3.13/site-packages/torchdata/stateful_dataloader/stateful_dataloader.py:379: UserWarning: 'set_vital' is deprecated, please do not call
  torch.set_vital("Dataloader", "enabled", "True")  # type: ignore[attr-defined]
/content/nemo-rl/nemo_rl/models/policy/lm_policy.py:355: UserWarning: TORCH_CUDA_ARCH_LIST is not set. This is needed if usi

## ✅ Step 4: First successful run (T4 + High-RAM, no checkpoint)
After enabling High-RAM, this one cell repeats Steps 2 and 3 on the fresh machine and runs 20 DPO steps. The flag guide is in the comments at the top of the cell.

**What to watch in the logs**
- Validation loss at step 0 is **0.6931 (= ln 2)**: the model cannot yet tell chosen from rejected.
- `accuracy`: fraction of pairs where the model now prefers the chosen answer (0.91 on validation at step 10).
- `rewards_chosen_mean` and `rewards_rejected_mean`: how far the model has moved away from the reference on each side. A growing gap means it is learning the preference.

No checkpoint is saved here, so the trained model is lost when the run ends (fixed in Step 5). One worker may be killed and restarted during startup; Ray retries it automatically.

In [11]:
# STEP 4: all-in-one on a fresh High-RAM T4: install, clone, build dataset, then run 20 DPO steps (no checkpoint)
# FLAG GUIDE (what each override on the command below does)
# policy.model_name / tokenizer.name  -> the small model we fine-tune (a frozen copy is the DPO reference)
# policy.precision=float32            -> the T4 has no native bf16
# policy.train_global_batch_size=8    -> 8 preference pairs per training step
# policy.train_micro_batch_size=1     -> 1 pair per GPU pass (saves memory)
# policy.max_total_sequence_length=512 -> longer examples are skipped
# policy.automodel_cfg.activation_checkpointing=true -> recompute activations to save GPU memory
# dpo.max_num_steps=20                -> train for 20 steps only (a demo)
# dpo.val_period=10, val_batches=4    -> validate at the start and every 10 steps
# ++data.*                            -> point NeMo RL at train.jsonl / val.jsonl (prompt, chosen, rejected)
# MPLBACKEND=Agg                      -> avoids the Colab matplotlib crash (see failed attempt 1)
# checkpointing.enabled=false         -> nothing saved (Step 5 fixes this)
!free -h | head -2; nvidia-smi --query-gpu=name --format=csv,noheader; pip -q install uv; cd /content && git clone -q --depth 1 https://github.com/NVIDIA-NeMo/RL.git nemo-rl && cd nemo-rl && git submodule update --init --depth 1 --recursive 2>&1 | tail -1; cd /content && python -c "import json,datasets as d; ds=d.load_dataset('Intel/orca_dpo_pairs',split='train').shuffle(seed=0).select(range(600)); [open(f,'w').write(''.join(json.dumps({'prompt':r['question'],'chosen':r['chosen'],'rejected':r['rejected']})+'\n' for r in ds.select(rng))) for f,rng in (('train.jsonl',range(512)),('val.jsonl',range(512,600)))]" && wc -l train.jsonl val.jsonl && cd /content/nemo-rl && MPLBACKEND=Agg uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=false ++data.train.dataset_name=BinaryPreferenceDataset ++data.train.data_path=/content/train.jsonl ++data.train.split=train ++data.validation.dataset_name=BinaryPreferenceDataset ++data.validation.data_path=/content/val.jsonl ++data.validation.split=train ++data.default.dataset_name=BinaryPreferenceDataset ++data.default.prompt_key=prompt ++data.default.chosen_key=chosen ++data.default.rejected_key=rejected 2>&1 | tee /content/dpo.log

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       9.2Gi       4.2Mi       2.7Gi        11Gi
Tesla T4
fatal: destination path 'nemo-rl' already exists and is not an empty directory.
    512 train.jsonl
     88 val.jsonl
    600 total
INFO:nemo_rl.transformers_compat:transformers_compat.py:70: Installed the Transformers compatibility helper is_torch_fx_available for legacy remote model code
2026-10-24 05:07:05,698	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
INFO:nemo_rl.distributed.virtual_cluster:virtual_cluster.py:547: Started local cluster with tag 'nrl_tag_ALL': {'CPU': 2.0, 'node:172.28.0.12': 1.0, 'GPU': 1.0, 'nrl_tag_ALL': 1.0, 'accelerator_type:T4': 1.0, 'node:__internal_head__': 1.0, 'object_store_memory': 3989429452.0, 'memory': 9308668724.0}
Loaded configuration from: /content/nemo-rl/examples/configs/dpo.yaml
Overrides: ['policy.model_name=Qwen/Qwen2.5-0

### ⚠️ Cleanup cell that backfired
`pkill -f` matched its own command line and killed the cell (shown as ^C). Ignore this cell.

In [ ]:
!pkill -9 -f "[r]un_dpo.py"; pkill -9 -f "[r]aylet"; pkill -9 -f "[g]cs_server"; sleep 3; nvidia-smi --query-gpu=memory.used --format=csv,noheader; cd /content/nemo-rl && rm -rf results && MPLBACKEND=Agg uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true policy.automodel_cfg.checkpoint.save_consolidated=final dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=true checkpointing.save_period=20 checkpointing.save_optimizer=false checkpointing.keep_top_k=1 ++data.train.dataset_name=BinaryPreferenceDataset ++data.train.data_path=/content/train.jsonl ++data.train.split=train ++data.validation.dataset_name=BinaryPreferenceDataset ++data.validation.data_path=/content/val.jsonl ++data.validation.split=train ++data.default.dataset_name=BinaryPreferenceDataset ++data.default.prompt_key=prompt ++data.default.chosen_key=chosen ++data.default.rejected_key=rejected > /content/dpo3.log 2>&1; tail -4 /content/dpo3.log; find /content/nemo-rl/results -maxdepth 6 | head -30

^C


## ✅ Step 5: Re-run with checkpoint saving
Same training as Step 4, with these changes so the model can be loaded afterwards:
- `checkpointing.enabled=true` and `checkpointing.save_period=20`: save at step 20.
- `checkpointing.save_optimizer=false`: skip optimizer state to save disk space.
- `policy.automodel_cfg.checkpoint.save_consolidated=final`: also write a regular Hugging Face style weights folder (the `consolidated` directory) that transformers can load.

Weights end up in `/content/nemo-rl/results/dpo/step_20/policy/weights/model/consolidated`.

In [ ]:
# STEP 5: same training, but save the model so we can load it in Step 6
# FLAG GUIDE (what each override on the command below does)
# policy.model_name / tokenizer.name  -> the small model we fine-tune (a frozen copy is the DPO reference)
# policy.precision=float32            -> the T4 has no native bf16
# policy.train_global_batch_size=8    -> 8 preference pairs per training step
# policy.train_micro_batch_size=1     -> 1 pair per GPU pass (saves memory)
# policy.max_total_sequence_length=512 -> longer examples are skipped
# policy.automodel_cfg.activation_checkpointing=true -> recompute activations to save GPU memory
# dpo.max_num_steps=20                -> train for 20 steps only (a demo)
# dpo.val_period=10, val_batches=4    -> validate at the start and every 10 steps
# ++data.*                            -> point NeMo RL at train.jsonl / val.jsonl (prompt, chosen, rejected)
# MPLBACKEND=Agg                      -> avoids the Colab matplotlib crash (see failed attempt 1)
# checkpointing.enabled=true, save_period=20 -> save a checkpoint at step 20
# checkpointing.save_optimizer=false  -> skip optimizer state (smaller, faster)
# ...checkpoint.save_consolidated=final -> also write a Hugging Face style weights folder
!nvidia-smi --query-gpu=memory.used --format=csv,noheader; cd /content/nemo-rl && rm -rf results && MPLBACKEND=Agg uv run examples/run_dpo.py policy.model_name=Qwen/Qwen2.5-0.5B-Instruct policy.tokenizer.name=Qwen/Qwen2.5-0.5B-Instruct policy.precision=float32 policy.train_global_batch_size=8 policy.train_micro_batch_size=1 policy.max_total_sequence_length=512 policy.automodel_cfg.activation_checkpointing=true policy.automodel_cfg.checkpoint.save_consolidated=final dpo.max_num_steps=20 dpo.val_period=10 dpo.val_batches=4 dpo.val_global_batch_size=4 checkpointing.enabled=true checkpointing.save_period=20 checkpointing.save_optimizer=false checkpointing.keep_top_k=1 ++data.train.dataset_name=BinaryPreferenceDataset ++data.train.data_path=/content/train.jsonl ++data.train.split=train ++data.validation.dataset_name=BinaryPreferenceDataset ++data.validation.data_path=/content/val.jsonl ++data.validation.split=train ++data.default.dataset_name=BinaryPreferenceDataset ++data.default.prompt_key=prompt ++data.default.chosen_key=chosen ++data.default.rejected_key=rejected > /content/dpo3.log 2>&1; tail -4 /content/dpo3.log; find /content/nemo-rl/results -maxdepth 6 | head -30

0 MiB
  • policy_training/submit_training_futures: 0.00s (0.0%)
  • policy_training/sharding_data: 0.00s (0.0%)
Max number of steps has been reached, stopping training early
Ray is not initialized. Cannot collect GPU metrics.
/content/nemo-rl/results
/content/nemo-rl/results/dpo
/content/nemo-rl/results/dpo/step_20
/content/nemo-rl/results/dpo/step_20/train_dataloader.pt
/content/nemo-rl/results/dpo/step_20/config.yaml
/content/nemo-rl/results/dpo/step_20/training_info.json
/content/nemo-rl/results/dpo/step_20/policy
/content/nemo-rl/results/dpo/step_20/policy/tokenizer
/content/nemo-rl/results/dpo/step_20/policy/tokenizer/vocab.json
/content/nemo-rl/results/dpo/step_20/policy/tokenizer/tokenizer.json
/content/nemo-rl/results/dpo/step_20/policy/tokenizer/merges.txt
/content/nemo-rl/results/dpo/step_20/policy/tokenizer/tokenizer_config.json
/content/nemo-rl/results/dpo/step_20/policy/tokenizer/chat_template.jinja
/content/nemo-rl/results/dpo/step_20/policy/weights
/content/nemo-rl/resul

## Step 6: Compare base vs DPO model
The next cell loads both models (the original from Hugging Face, the DPO one from the checkpoint above) and asks the same five questions with greedy decoding (no randomness), so any difference comes from the training. Answers are saved to `/content/compare_out.txt`.

In [ ]:
# Load the original model and the DPO-trained checkpoint, ask both the same questions
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM
ck='/content/nemo-rl/results/dpo/step_20/policy/weights/model/consolidated'  # DPO weights saved in Step 5
print(os.listdir(ck))
tok=AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
def load(p): return AutoModelForCausalLM.from_pretrained(p, torch_dtype=torch.float32).cuda().eval()
base=load('Qwen/Qwen2.5-0.5B-Instruct'); dpo=load(ck)
qs=['What is the capital of Australia? Explain briefly.',
'Explain in two sentences why the sky is blue.',
'A train travels 60 km in 45 minutes. What is its average speed in km/h? Show your steps.',
'Write a short, polite email declining a meeting invitation.',
'Is it safe to leave cooked rice out overnight? Answer briefly.']
def ask(m,q):
    ids=tok.apply_chat_template([{'role':'user','content':q}],add_generation_prompt=True,return_tensors='pt',return_dict=True).to('cuda')
    out=m.generate(**ids,max_new_tokens=170,do_sample=False)  # greedy = repeatable answers
    return tok.decode(out[0][ids['input_ids'].shape[1]:],skip_special_tokens=True).strip()
res=[]
for q in qs:
    res.append('Q: '+q+'\n--BASE: '+ask(base,q)+'\n--DPO : '+ask(dpo,q)+'\n'+'='*70)
open('/content/compare_out.txt','w').write('\n'.join(res))
print('done', len(res))


## Step 7: Read the results
Prints the saved side-by-side answers. Summary of what they show is at the bottom of this notebook.

In [ ]:
# STEP 7: print the saved base-vs-DPO answers
!cat /content/compare_out.txt

Q: What is the capital of Australia? Explain briefly.
--BASE: The capital of Australia is Canberra. It is located in the northeastern part of the country and serves as the official residence of the Prime Minister of Australia. Canberra is known for its modern architecture, beautiful landscapes, and rich cultural heritage. The city also hosts various international events and conferences.
--DPO : The capital of Australia is Canberra. This city serves as the governmental and administrative center of the country, known for its modern infrastructure, cultural institutions, and significant role in international affairs.
Q: Explain in two sentences why the sky is blue.
--BASE: The sky appears blue because it reflects sunlight and scatters other colors of light, particularly red, orange, yellow, green, and blue. These colors are scattered more than white or black, creating the illusion of a uniform color for the entire spectrum of visible light.
--DPO : The sky appears blue due to the scatteri

# DPO demo on a T4: results and explanation

Trained Qwen2.5-0.5B-Instruct with DPO (20 steps, 512 preference pairs from Intel/orca_dpo_pairs, float32, checkpoint saved), then asked the base model and the DPO-trained model five questions each. Both used greedy decoding, so answers are repeatable.

| Question | Base model | DPO model | Better? |
|---|---|---|---|
| Capital of Australia, briefly | Canberra, but says it's in the "northeastern part" of the country (wrong) and adds filler | Canberra, shorter, no factual slip, a bit generic | DPO |
| Why is the sky blue (2 sentences) | Says the sky "reflects" sunlight and scatters red, orange, yellow, green and blue (wrong) | Correctly points to scattering by atmospheric particles, but wrongly names "water vapor and nitrogen oxides" | DPO slightly, both flawed |
| Train: 60 km in 45 min, show steps | Long LaTeX walkthrough, cut off at my 170-token limit before the answer | Converts to 0.75 h, divides, reaches 80 km/h, which is correct | DPO |
| Polite email declining a meeting | Odd template that says "As an AI language model, I cannot participate in meetings" | Same template, minus that odd AI line; still reads like a rejection letter | DPO slightly, both weak |
| Cooked rice overnight, briefly | Says no, then wrongly gives 40-140°F as the ideal storage range (that's the bacterial danger zone) | Says no, one clean sentence | DPO |

**What changed:** the DPO model is more concise, more polished and formal, and follows "briefly" and "two sentences" better. It also dropped a few of the base model's odd or wrong filler details.

**What didn't change:** it isn't smarter. Both models are still a 0.5B model that gets some facts wrong (the sky-blue answer is wrong in both, in different ways). DPO shifted the style toward the preferred answers, which in this dataset are the longer, more careful ones written by a strong model (GPT-4), rather than adding knowledge.

**Small sample:** five questions at one answer each is a quick sanity check, not a benchmark. The 20-step training also drifted the reward values a lot, though the outputs stayed coherent.

**Where things are:** the trained weights are in the Colab runtime at `/content/nemo-rl/results/dpo/step_20/policy/weights/model/consolidated`, and the full side-by-side text is in `/content/compare_out.txt`. They will be lost when the runtime is deleted, so download them if you want to keep them.
